# PRoViT Baseline — Decoder Matrix, aligned to WARP's $W^{(0)}$

1. Run §1–§6 top to bottom (defines everything, doesn't start experiments).
2. `run_matrix()` under "Run" iterates `RUN_MATRIX` — six (model, task) groups × 3 PRoViT variants × 7 gap thresholds — and writes one JSON per group.
3. The diagnostic and summary-table cells can be run at any time; they read the saved JSON and never recompute.
4. Every group is checked for alignment against WARP. Any `⚠` means that group's numbers must **not** be placed beside the WARP numbers.


### Before running

**Prerequisite**: `decoder_experiments.ipynb` must have been run first. This notebook reads the Stage-A / GSN-FT checkpoints it wrote to `ckpt_decoder/`, and the caches and results it wrote to `results_decoder/`. §2 prints `OK` or `MISSING` for both directories.

**Gurobi WLS license** — needed by the `LP` and `FT+LP` variants only; `FT` runs without it. The license is read from an environment variable and is never written into the notebook. Set it in your own session before running:

```python
import os
os.environ['GUROBI_WLS_LICENSE'] = '''# Gurobi WLS license file
WLSACCESSID=...
WLSSECRET=...
LICENSEID=...
'''
```

Free academic licenses: https://www.gurobi.com/academia/


## §1. Install & Gurobi license

In [ ]:
# Install packages + Gurobi WLS license
!pip install -q transformers datasets scipy gurobipy

# This notebook is pure text classification and never touches torchvision.
# Recent `datasets` versions, once torchvision is importable, probe
# torchvision.io.VideoReader for a type check; recent torchvision has
# deprecated/removed VideoReader, so that probe raises. Removing torchvision
# keeps it from firing.
!pip uninstall -y torchvision -q

import os

# Read the license from the environment — never hard-coded here.
GUROBI_LIC = os.environ.get('GUROBI_WLS_LICENSE')
if GUROBI_LIC:
    with open('/content/gurobi.lic', 'w') as f:
        f.write(GUROBI_LIC)
    os.environ['GRB_LICENSE_FILE'] = '/content/gurobi.lic'
else:
    print('[gurobi] GUROBI_WLS_LICENSE not set — set your own WLS license '
          '(see "Before running"), otherwise the LP variants cannot run.')

try:
    import gurobipy as gp
    from gurobipy import GRB
    _env = gp.Env()
    HAS_GUROBI = True
    print('[gurobi] WLS license OK')
except Exception as e:
    HAS_GUROBI = False
    print(f'[gurobi] unavailable ({e}) → LP variants cannot run '
          f'(the paper-faithful objective needs Gurobi)')

## §2. Determinism & paths

In [ ]:
# Determinism settings + paths
# These must match the WARP decoder notebook exactly. A different numeric setup
# changes the logits bit-for-bit, hence the classifier-head inputs, and the
# alignment checks in §4 would fail.
import os
os.environ.setdefault('PYTHONHASHSEED', '0')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import json, time, pickle, gc, random, warnings
import numpy as np
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')

torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False    # matches WARP
torch.backends.cudnn.allow_tf32       = False
torch.set_float32_matmul_precision('highest')

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Must be the same DRIVE_SUBDIR the decoder notebook used, so that the
# CKPT_DIR / WARP_RESULT it wrote can be read back.
DRIVE_SUBDIR = 'WARP Stuff'  # shared with the encoder and decoder notebooks

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = f'/content/drive/MyDrive/{DRIVE_SUBDIR}'
except Exception:
    # Not on Colab (local / other notebook service): fall back to a relative path
    BASE_DIR = f'./{DRIVE_SUBDIR}_local'
    print(f'[info] not on Colab or mount failed, using local path: {BASE_DIR}')

CKPT_DIR    = f'{BASE_DIR}/ckpt_decoder'        # Stage-A / GSN-FT artifacts
WARP_RESULT = f'{BASE_DIR}/results_decoder'     # cache_* / result_* from the decoder notebook
RESULT_DIR  = f'{BASE_DIR}/results_provit_matrix'
os.makedirs(RESULT_DIR, exist_ok=True)
for p in (CKPT_DIR, WARP_RESULT):
    print(f'{"OK     " if os.path.isdir(p) else "MISSING"} {p}')

print(f'device={DEVICE}  TF32=off  OMP_NUM_THREADS={os.environ["OMP_NUM_THREADS"]}')

## §3. Config

**Only this section** needs editing to control which (model, task) combinations run.

In [ ]:
# Run matrix + hyperparameters
RUN_MATRIX = [
    ('gpt2',         'sst2'),
    ('gpt2',         'rte'),
    ('gpt2-medium',  'sst2'),
    ('gpt2-medium',  'rte'),
    ('qwen2.5-0.5b', 'sst2'),
    ('qwen2.5-0.5b', 'rte'),
]

BASE_IDS = {
    'gpt2':         'gpt2',
    'gpt2-medium':  'gpt2-medium',
    'qwen2.5-0.5b': 'Qwen/Qwen2.5-0.5B',
}
# Matches the WARP registry: gpt2/sst2 uses a ready-made HF checkpoint, the rest
# use the Stage-A artifacts written by the decoder notebook.
TASK_CKPT = {('gpt2', 'sst2'): 'PavanNeerudu/gpt2-finetuned-sst2'}

MAX_LEN     = {'sst2': 128, 'rte': 256}     # matches WARP TASK_CONFIG
N_REMAIN_QP = 200                            # matches WARP
CACHE_BS    = 16

# WARP curation parameters, used to rebuild exactly the same S
REPAIR_GAP_FLOOR_MODE = 'relative'
REPAIR_GAP_FLOOR_REL  = 1.5

# PRoViT hyperparameters (from the original script, faithful to Nawas et al. 2024)
PROVIT_EPS        = 0.01
PROVIT_LP_TIMEOUT = 7200.0
PROVIT_FT_KWARGS  = dict(lr=1e-3, gamma=0.995, batch_size=10,
                         validation_interval=5, max_epochs=5000,
                         time_limit=3600.0)
# None = WARP's full curated S (the main-table comparison); the remaining values
# are the gap-threshold sweep reported in the appendix.
THRESHOLDS = [None, -1.0, -2.0, -3.0, -5.0, -7.0, -10.0]

VARIANTS = ['LP', 'FT', 'FT+LP']

print(f'{len(RUN_MATRIX)} groups × {len(VARIANTS)} variants × {len(THRESHOLDS)} thresholds')

## §4. Load $W^{(0)}$ and align to WARP's sample sets

`gsnft.pt` is the complete `model.state_dict()` after GSN-FT, including the score head. Loading the Stage-A checkpoint and then applying it reproduces, bit for bit, the state WARP was in immediately before entering the QP. When the file is absent, WARP had skipped GSN-FT because $\kappa_0$ already met `KAPPA_TARGET`, and the Stage-A model *is* $W^{(0)}$.

Two consistency checks run for every group, and **both must pass** before that group's numbers may be placed beside WARP's:

1. **Predictions** — re-encoding the same texts through the reconstructed $W^{(0)}$ must reproduce every prediction stored in WARP's cache.
2. **Curated $|S|$** — rebuilding WARP's curation rule (a relative gap floor at $-1.5 \times$ the median remain gap) from the recomputed gaps must yield the same repair-set size WARP reported.


In [ ]:
# Load W(0) + rebuild WARP's sample sets
from transformers import AutoTokenizer, AutoModelForSequenceClassification


def load_w0_model(model_key, task):
    """Load the Stage-A checkpoint and apply gsnft.pt => exactly WARP's W(0)."""
    src = TASK_CKPT.get((model_key, task), f'{CKPT_DIR}/{model_key}_{task}_taskft')
    if src.startswith('/') and not os.path.isdir(src):
        raise FileNotFoundError(f'Stage-A artifact not found: {src}')
    tok = AutoTokenizer.from_pretrained(BASE_IDS[model_key])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(
        src, num_labels=2, torch_dtype=torch.float32).to(DEVICE)
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tok.pad_token_id

    gsn_path = f'{CKPT_DIR}/{model_key}_{task}_gsnft.pt'
    if os.path.exists(gsn_path):
        model.load_state_dict(torch.load(gsn_path, map_location=DEVICE))
        gsn_applied = True
    else:
        gsn_applied = False
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    print(f'  [W(0)] {src}   GSN-FT state: '
          f'{"applied" if gsn_applied else "none (WARP skipped it)"}')
    return model, tok, model.score, gsn_applied


@torch.no_grad()
def cache_head_inputs(model, tok, CLF, texts, max_len, W, b, desc='', bs=CACHE_BS):
    """Capture the classifier-head input v_cls per text, and predict with (W, b)."""
    cap = {}
    hk = CLF.register_forward_hook(
        lambda _m, inp, _o: cap.__setitem__('x', inp[0].detach()))
    V, P = [], []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i + bs]
        enc = tok(chunk, return_tensors='pt', truncation=True,
                  max_length=max_len, padding=True)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        model(**enc)
        x = cap['x']
        if x.dim() == 3:                      # decoder: take the last non-pad token
            mask = enc['attention_mask']
            last = mask.shape[1] - 1 - mask.flip(1).float().argmax(1)
            x = x[torch.arange(x.shape[0], device=x.device), last.long()]
        Vb = x.cpu().numpy().astype(np.float64)
        for v in Vb:
            V.append(v); P.append(int(np.argmax(W @ v + b)))
        if desc and (i // bs) % 20 == 19:
            print(f'    {desc} {min(i + bs, len(texts))}/{len(texts)}')
    hk.remove()
    return np.stack(V), np.array(P)


def build_aligned_sets(model_key, task):
    """Read WARP's cache, encode the same samples, and verify both checks."""
    cpath = f'{WARP_RESULT}/cache_{model_key}_{task}.pkl'
    rpath = f'{WARP_RESULT}/result_{model_key}_{task}.pkl'
    if not os.path.exists(cpath):
        raise FileNotFoundError(f'WARP cache not found: {cpath}')
    with open(cpath, 'rb') as f:
        D = pickle.load(f)
    R = pickle.load(open(rpath, 'rb')) if os.path.exists(rpath) else None

    model, tok, CLF, gsn_applied = load_w0_model(model_key, task)
    W0 = CLF.weight.detach().cpu().numpy().astype(np.float64).copy()
    has_bias = CLF.bias is not None
    b0 = (CLF.bias.detach().cpu().numpy().astype(np.float64).copy()
          if has_bias else np.zeros(W0.shape[0]))
    ml = MAX_LEN[task]
    print(f'  head: {W0.shape}  bias={has_bias}')

    def enc_set(items, desc):
        V, P = cache_head_inputs(model, tok, CLF, [s['text'] for s in items],
                                 ml, W0, b0, desc=desc)
        return [{'text': s['text'], 'gt': int(s['gt']), 'v': v,
                 'pred': int(p), 'warp_pred': int(s['pred'])}
                for s, v, p in zip(items, V, P)]

    sets = {
        'repair_all': enc_set(D['repair_set'], 'repair'),
        'remain':     enc_set(D['remain_set'][:N_REMAIN_QP], 'remain'),
        'clean':      enc_set(D['clean_set'], 'clean'),
    }
    sets['attacks'] = {m: enc_set(c, m) for m, c in sorted(D['test_caches'].items())}

    # Check 1: predictions must match WARP's cache exactly
    allv = (sets['repair_all'] + sets['remain'] + sets['clean']
            + [s for c in sets['attacks'].values() for s in c])
    n_mis = sum(1 for s in allv if s['pred'] != s['warp_pred'])
    tag1 = 'OK' if n_mis == 0 else f'⚠ {n_mis}/{len(allv)} mismatched'
    print(f'  [check 1] predictions match WARP cache: {tag1}')

    # Check 2: rebuilding the curation must give the same |S|
    def sgap(s, W=W0, b=b0):
        lo = W @ s['v'] + b
        return float(lo[s['gt']] - lo[1 - s['gt']])

    rem_gaps = np.array([abs(sgap(s)) for s in sets['remain']])
    scale = float(np.median(rem_gaps))
    floor = (-REPAIR_GAP_FLOOR_REL * scale
             if REPAIR_GAP_FLOOR_MODE == 'relative' else None)
    init_gaps = np.array([sgap(s) for s in sets['repair_all']])
    keep = init_gaps >= floor if floor is not None else np.ones(len(init_gaps), bool)
    sets['repair_curated'] = [s for s, k in zip(sets['repair_all'], keep) if k]
    sets['repair_gaps'] = init_gaps

    n_warp = R['set_sizes']['S'] if R else None
    n_here = len(sets['repair_curated'])
    tag2 = ('OK' if n_warp is None or n_here == n_warp
            else f'⚠ WARP={n_warp} here={n_here}')
    print(f'  [check 2] curated |S|={n_here}'
          + (f' (WARP={n_warp}) {tag2}' if n_warp else '')
          + f'   floor={floor:.2f} scale={scale:.2f}')

    meta = {'W0': W0, 'b0': b0, 'has_bias': has_bias,
            'gsn_applied': gsn_applied, 'pred_mismatch': n_mis,
            'n_S_warp': n_warp, 'n_S_here': n_here,
            'S_match': (n_warp is None or n_here == n_warp),
            'warp_W0_hash': (R or {}).get('repro', {}).get('W0_hash')}

    del model
    gc.collect(); torch.cuda.empty_cache()
    return sets, meta

print('Alignment utilities defined.')

## §5. PRoViT core (LP / FT / FT+LP)

Carried over unchanged from the original script, faithful to Nawas et al. (2024). All three variants edit **only** the final classification head:

- **LP** — paper Eq. 3. Minimizes the paper's `norm_ub` objective ($\ell_1/n + \ell_\infty$) subject to a margin $\varepsilon$ on every repair sample. Needs Gurobi.
- **FT** — paper §3.1. Fine-tunes the head until repair accuracy reaches 100%.
- **FT+LP** — paper §3.3. Runs FT first; if it converges, returns it, otherwise warm-starts the LP from one FT epoch.

For a binary task, $K$ is the set of labels appearing in the repair set.


In [ ]:
# PRoViT core: LP / FT / FT+LP
def _norm_ub_gurobi(mvar, m):
    """The paper's norm_ub objective: L1/n + Linf."""
    flat = mvar.reshape(-1); n = mvar.size
    Z = m.addMVar(shape=(n,), lb=0.0)
    for v, z in zip(flat, Z):
        m.addConstr(v <= z); m.addConstr(-v <= z)
    z_max = m.addVar()
    m.addConstr(z_max >= Z)
    return (Z.sum() / n) + z_max


def provit_lp_repair(repair_cache, W0, b0, has_bias, W_start=None, b_start=None,
                     *, eps=PROVIT_EPS, verbose=False, time_limit=PROVIT_LP_TIMEOUT):
    """Paper Eq. 3, restricted to the final classification head."""
    W = W0.copy() if W_start is None else W_start.copy()
    b = b0.copy() if b_start is None else b_start.copy()
    n_lab, d = W.shape
    K = sorted({int(s['gt']) for s in repair_cache})
    if not HAS_GUROBI:
        raise RuntimeError('Gurobi required (see §1).')
    m = gp.Model('provit_lp', env=_env)
    m.Params.OutputFlag = 1 if verbose else 0
    m.Params.Crossover, m.Params.Method = 0, 2
    m.Params.Threads = os.cpu_count() or 4
    m.Params.BarConvTol = 1e-4
    m.Params.TimeLimit = float(time_limit)
    W_K = W[K, :].copy(); b_K = b[K].copy()
    dW = m.addMVar(shape=W_K.shape, lb=-10.0, ub=10.0)
    db = m.addMVar(shape=b_K.shape, lb=-10.0, ub=10.0) if has_bias else None
    lab2loc = {lab: i for i, lab in enumerate(K)}
    t0 = time.perf_counter()
    for s in repair_cache:
        v = s['v']; l = lab2loc[int(s['gt'])]
        sym = (W_K @ v) + (dW @ v) + (b_K + db if has_bias else b_K)
        for i in range(len(K)):
            if i != l:
                m.addConstr(sym[l] >= sym[i] + eps)
        if len(K) < n_lab:
            m.addConstr(sym[l] >= float(np.max(W @ v + b)) + eps)
    obj = _norm_ub_gurobi(dW, m)
    if has_bias:
        obj = obj + _norm_ub_gurobi(db, m)
    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()
    st = time.perf_counter() - t0
    if m.status != GRB.OPTIMAL:
        return None, st, False
    W_new, b_new = W.copy(), b.copy()
    for i_loc, lab in enumerate(K):
        W_new[lab] = W_K[i_loc] + dW.X[i_loc]
        if has_bias:
            b_new[lab] = b_K[i_loc] + db.X[i_loc]
    return (W_new, b_new), st, True


def provit_ft_repair(repair_cache, W0, b0, has_bias, *, lr, gamma, batch_size,
                     max_epochs, validation_interval, time_limit, verbose=False):
    """Paper §3.1: fine-tune the head only, until repair accuracy hits 100%."""
    n_lab, d = W0.shape
    head = nn.Linear(d, n_lab, bias=has_bias).to(DEVICE)
    with torch.no_grad():
        head.weight.copy_(torch.from_numpy(W0).float())
        if has_bias:
            head.bias.copy_(torch.from_numpy(b0).float())
    V = torch.tensor(np.stack([s['v'] for s in repair_cache]),
                     dtype=torch.float32, device=DEVICE)
    y = torch.tensor([int(s['gt']) for s in repair_cache], device=DEVICE)
    opt   = torch.optim.Adam(head.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=gamma)
    crit  = nn.CrossEntropyLoss()
    n = V.shape[0]; t0 = time.perf_counter(); converged = False
    for epoch in range(max_epochs):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            crit(head(V[idx]), y[idx]).backward(); opt.step()
        sched.step()
        if epoch > 0 and epoch % validation_interval == 0:
            with torch.no_grad():
                acc = float((head(V).argmax(1) == y).float().mean())
            if verbose: print(f'    ep{epoch} acc={acc:.4f}')
            if acc >= 1.0 - 1e-9:
                converged = True; break
        if time.perf_counter() - t0 > time_limit:
            break
    el = time.perf_counter() - t0
    if not converged:
        print(f'    [FT] did not reach 100% ({el:.1f}s)')
        return None, el
    b_new = (head.bias.detach().cpu().numpy().astype(np.float64)
             if has_bias else np.zeros(n_lab))
    return (head.weight.detach().cpu().numpy().astype(np.float64), b_new), el


def provit_ft_plus_lp_repair(repair_cache, W0, b0, has_bias, *, eps=PROVIT_EPS):
    """Paper §3.3: FT first; if it converges return it, else warm-start the LP."""
    out, _ = provit_ft_repair(repair_cache, W0, b0, has_bias, **PROVIT_FT_KWARGS)
    if out is not None:
        return out, {'used_lp': False}
    warm = dict(PROVIT_FT_KWARGS); warm['max_epochs'] = 1; warm['time_limit'] = 60.0
    ft1 = provit_ft_repair(repair_cache, W0, b0, has_bias, **warm)[0]
    W_s, b_s = ft1 if ft1 is not None else (W0, b0)
    lp_out, _, ok = provit_lp_repair(repair_cache, W0, b0, has_bias, W_s, b_s, eps=eps)
    return (lp_out if ok else None), {'used_lp': True}

print('PRoViT core defined.')

## §6. Evaluation + runner

In [ ]:
# Evaluation + runner
def _acc(items, W, b):
    if not items: return float('nan')
    V = np.stack([s['v'] for s in items])
    gt = np.array([s['gt'] for s in items])
    return float(np.mean(np.argmax(V @ W.T + b, axis=1) == gt))


def evaluate(sets, W, b, repair_subset):
    per = {m: _acc(c, W, b) for m, c in sets['attacks'].items()}
    return {'repair_acc': _acc(repair_subset, W, b) * 100,
            'remain_acc': _acc(sets['remain'], W, b) * 100,
            'clean_acc':  _acc(sets['clean'], W, b) * 100,
            'gen_mean':   float(np.mean(list(per.values()))) * 100,
            'gen': {m: v * 100 for m, v in per.items()}}


def fmt(ev, extra=''):
    return (f"repair={ev['repair_acc']:5.1f}%  remain={ev['remain_acc']:5.1f}%  "
            f"clean={ev['clean_acc']:5.1f}%  atk={ev['gen_mean']:5.1f}%  {extra}")


def run_one(model_key, task, force=False):
    out_path = f'{RESULT_DIR}/provit_{model_key}_{task}.json'
    if os.path.exists(out_path) and not force:
        print(f'[skip] {out_path} already exists'); return json.load(open(out_path))

    print(f'\n{"#"*78}\n#  PRoViT  {model_key} × {task}\n{"#"*78}')
    sets, meta = build_aligned_sets(model_key, task)
    W0, b0, has_bias = meta['W0'], meta['b0'], meta['has_bias']
    S_full = sets['repair_curated']

    # gaps of the curated subset, used by the threshold sweep
    def sg(s):
        lo = W0 @ s['v'] + b0
        return float(lo[s['gt']] - lo[1 - s['gt']])
    gaps_c = np.array([sg(s) for s in S_full])

    print(f'\nOriginal W(0): {fmt(evaluate(sets, W0, b0, S_full))}')

    RES = {'model_key': model_key, 'task': task,
           'gsn_applied': meta['gsn_applied'],
           'pred_mismatch': meta['pred_mismatch'],
           'n_S_warp': meta['n_S_warp'], 'n_S_here': meta['n_S_here'],
           'S_match': bool(meta['S_match']),
           'warp_W0_hash': meta['warp_W0_hash'],
           'n_remain': len(sets['remain']), 'n_clean': len(sets['clean']),
           'n_atk': sum(len(c) for c in sets['attacks'].values()),
           'w0_eval': evaluate(sets, W0, b0, S_full),
           'variants': {}}

    for variant in VARIANTS:
        rows = []
        print(f'\n{"="*78}\n{model_key}/{task}  PRoViT-{variant}\n{"="*78}')
        for T in THRESHOLDS:
            if T is None:
                subset, lab = S_full, 'full(=WARP S)'
            else:
                subset = [s for s, g in zip(S_full, gaps_c) if g >= T]
                lab = f'T={T:>5}'
            n = len(subset)
            if n == 0:
                print(f'{lab:<14} |S|=0 (skip)'); continue
            t0 = time.perf_counter()
            try:
                if variant == 'LP':
                    out, _, ok = provit_lp_repair(subset, W0, b0, has_bias)
                elif variant == 'FT':
                    out, _ = provit_ft_repair(subset, W0, b0, has_bias,
                                              **PROVIT_FT_KWARGS)
                    ok = out is not None
                else:
                    out, _info = provit_ft_plus_lp_repair(subset, W0, b0, has_bias)
                    ok = out is not None
            except Exception as e:
                print(f'{lab:<14} |S|={n:>4}  EXCEPTION {type(e).__name__}: {e}')
                rows.append({'T': T, 'n_S': n, 'status': 'exception'}); continue
            dt = time.perf_counter() - t0
            if not ok:
                print(f'{lab:<14} |S|={n:>4}  FAILED ({dt:.1f}s)')
                rows.append({'T': T, 'n_S': n, 'status': 'failed', 't': dt}); continue
            Wn, bn = out
            ev = evaluate(sets, Wn, bn, subset)
            dW = float(np.linalg.norm(Wn - W0, 'fro'))
            rows.append({'T': T, 'n_S': n, 'status': 'ok', **ev,
                         'dW_fro': dW, 't': dt})
            print(f'{lab:<14} |S|={n:>4}  ' + fmt(ev, f'|dW|F={dW:.3f} {dt:.1f}s'))
        RES['variants'][variant] = rows

    with open(out_path, 'w') as f:
        json.dump(RES, f, indent=2, default=str)
    print(f'\n[saved] {out_path}')
    del sets; gc.collect(); torch.cuda.empty_cache()
    return RES


def run_matrix(force=False):
    out = {}
    for mk, tk in RUN_MATRIX:
        try:
            out[(mk, tk)] = run_one(mk, tk, force=force)
        except Exception as e:
            import traceback as _tb
            msg = f'{type(e).__name__}: {e}'
            _tb.clear_frames(e.__traceback__); del e
            gc.collect(); torch.cuda.empty_cache()
            print(f'[ERROR] {mk}/{tk}: {msg}')
    print(f'\ncompleted {len(out)}/{len(RUN_MATRIX)} groups')
    return out

print('Runner defined.')


## Run

The six groups run in sequence. The `LP` variant can be slow when `|S|` is large (Gurobi barrier); the `FT` variant runs up to 5000 epochs. Budget roughly 10–40 minutes per group. Results are written to `results_provit_matrix/provit_<model>_<task>.json`; rerunning skips groups whose JSON already exists unless `force=True`.


In [ ]:
res = run_matrix()

## Diagnostic — majority-class baseline of the attack sets

The attack sets are label-imbalanced, so any accuracy near the majority-class rate carries no information. This is the reference line for reading the collapse check below.


In [ ]:
# Attack-set label distribution + majority-class baseline
import pickle, collections
for mk, tk in RUN_MATRIX:
    D = pickle.load(open(f'{WARP_RESULT}/cache_{mk}_{tk}.pkl','rb'))
    pool = [s for c in D['test_caches'].values() for s in c]
    cnt = collections.Counter(s['gt'] for s in pool)
    maj = max(cnt.values())/len(pool)*100
    print(f'{mk}/{tk:<5} n={len(pool):>5}  gt distribution={dict(cnt)}  '
          f'majority-class baseline={maj:.1f}%')

### Collapse check — label inversion vs. robustness

**Hypothesis.** A method that may only edit the last layer, faced with an `S` consisting entirely of samples $W^{(0)}$ gets *wrong*, has one cheap solution available: flip the decision boundary wholesale — everything in `S` correct, everything else wrong.

**Test.** Measure the inversion rate, i.e. the fraction of samples whose PRoViT prediction is the opposite of the $W^{(0)}$ prediction. If it approaches 100%, the reported AtkGen is merely $100 - \mathrm{AtkGen}(W^{(0)})$ and carries no robustness meaning.

Reading the table:

- `remain < 90%` ⇒ preservation has already collapsed; that row must not enter an AtkGen comparison (marked †).
- $\lvert \mathrm{AtkGen} - (100 - \mathrm{AtkGen}(W^{(0)})) \rvert < 8\,\mathrm{pp}$ ⇒ the solution is essentially a label inversion of $W^{(0)}$: its "high AtkGen" comes from getting everything previously correct wrong, not from robustness.
- Majority-class baselines are 58.2% (SST-2) and 58.1% (RTE). Collapsed rows tend to sit near or above that as well, but the inversion relation explains the observed values more precisely.


In [ ]:
# Collapse check: inversion rate = fraction where PRoViT disagrees with W(0)
import os, json, pickle
import numpy as np

print('=' * 104)
print('Collapse check — inversion rate = fraction where PRoViT disagrees with W(0)')
print('=' * 104)
print(f'{"setting":<20}{"variant":<9}{"remain":>8}{"clean":>8}{"atk":>8}'
      f'{"100-W(0)atk":>13}{"inv(atk)":>10}{"inv(clean)":>12}  verdict')

rows_md = ['| Setting | Variant | Remain | Clean | AtkGen | 100−W(0) AtkGen '
           '| Inversion (atk) | Verdict |', '|---|---|---|---|---|---|---|---|']

for mk, tk in RUN_MATRIX:
    jp = f'{RESULT_DIR}/provit_{mk}_{tk}.json'
    cp = f'{WARP_RESULT}/cache_{mk}_{tk}.pkl'
    if not (os.path.exists(jp) and os.path.exists(cp)):
        continue
    R = json.load(open(jp))
    D = pickle.load(open(cp, 'rb'))

    # W(0) predictions on the attack sets, already stored in the cache
    atk_pool   = [s for m in sorted(D['test_caches']) for s in D['test_caches'][m]]
    w0_atk_pred = np.array([s['pred'] for s in atk_pool])
    w0_atk_acc  = 100.0 * np.mean([s['correct'] for s in atk_pool])

    # Recovering PRoViT's predictions would need the model again; use the
    # equivalent algebraic relation instead — under a complete inversion,
    #   acc_provit = 100 - acc_W(0). We compare against that directly.
    for variant in VARIANTS:
        full = next((r for r in R['variants'].get(variant, [])
                     if r.get('T') is None), None)
        if full is None or full.get('status') != 'ok':
            continue
        inv_pred = 100.0 - w0_atk_acc          # AtkGen implied by a complete inversion
        gap_to_inv = abs(full['gen_mean'] - inv_pred)
        collapsed = full['remain_acc'] < 90.0
        if collapsed and gap_to_inv < 8.0:
            verdict = 'inversion (AtkGen meaningless)'
        elif collapsed:
            verdict = 'partial collapse'
        else:
            verdict = 'valid'
        print(f'{mk+"/"+tk:<20}{variant:<9}{full["remain_acc"]:>7.1f}%'
              f'{full["clean_acc"]:>7.1f}%{full["gen_mean"]:>7.1f}%'
              f'{inv_pred:>12.1f}%{gap_to_inv:>9.1f}pp'
              f'{100.0 - full["clean_acc"]:>11.1f}%  {verdict}')
        rows_md.append(f'| {mk}/{tk} | PRoViT-{variant} | '
                       f'{full["remain_acc"]:.1f}% | {full["clean_acc"]:.1f}% | '
                       f'{full["gen_mean"]:.1f}% | {inv_pred:.1f}% | '
                       f'{gap_to_inv:.1f}pp | {verdict} |')
    print('-' * 104)


with open(f'{RESULT_DIR}/PROVIT_COLLAPSE.md', 'w') as f:
    f.write('\n'.join(rows_md) + '\n')
print(f'[saved] {RESULT_DIR}/PROVIT_COLLAPSE.md')

## Summary tables

The `full(=WARP S)` row is the fair main-table comparison — PRoViT repairs exactly the same sample set as WARP. The threshold sweep is appendix material.


In [ ]:
# Summary tables
import glob, pickle

rows = []
print('=' * 108)
print('Consistency check (both columns must be OK, else the group is not comparable to WARP)')
print('=' * 108)
print(f'{"setting":<22}{"GSN-FT":>9}{"pred mism.":>12}{"|S| here":>10}'
      f'{"|S| WARP":>10}{"match":>8}   WARP W(0) hash')
for mk, tk in RUN_MATRIX:
    p = f'{RESULT_DIR}/provit_{mk}_{tk}.json'
    if not os.path.exists(p):
        print(f'{mk+"/"+tk:<22}   (not run)'); continue
    R = json.load(open(p))
    print(f'{mk+"/"+tk:<22}{str(R["gsn_applied"]):>9}{R["pred_mismatch"]:>12}'
          f'{R["n_S_here"]:>10}{str(R["n_S_warp"]):>10}'
          f'{("OK" if R["S_match"] else "⚠"):>8}   {R.get("warp_W0_hash")}')

print('\n' + '=' * 108)
print('Main-table comparison — PRoViT repairs exactly the same curated S as WARP')
print('=' * 108)
print(f'{"setting":<22}{"method":<16}{"|S|":>6}{"Repair":>9}{"Remain":>9}'
      f'{"General":>9}{"AtkGen":>9}')
md_rows = ['| Setting | Method | \\|S\\| | Repair | Remain | General | AtkGen |',
           '|---|---|---|---|---|---|---|']
for mk, tk in RUN_MATRIX:
    p = f'{RESULT_DIR}/provit_{mk}_{tk}.json'
    wp = f'{WARP_RESULT}/result_{mk}_{tk}.pkl'
    if not os.path.exists(p): continue
    R = json.load(open(p))
    # the WARP row
    if os.path.exists(wp):
        Wr = pickle.load(open(wp, 'rb')); e = Wr['evals']['after']
        print(f'{mk+"/"+tk:<22}{"WARP (ours)":<16}{Wr["set_sizes"]["S"]:>6}'
              f'{e["repair_acc"]:>8.1f}%{e["remain_acc"]:>8.1f}%'
              f'{e["clean_acc"]:>8.1f}%{e["gen_mean"]:>8.1f}%')
        md_rows.append(f'| {mk}/{tk} | WARP (ours) | {Wr["set_sizes"]["S"]} | '
                       f'{e["repair_acc"]:.1f}% | {e["remain_acc"]:.1f}% | '
                       f'{e["clean_acc"]:.1f}% | {e["gen_mean"]:.1f}% |')
    for variant in VARIANTS:
        full = next((r for r in R['variants'].get(variant, [])
                     if r.get('T') is None), None)
        if full is None: continue
        if full.get('status') != 'ok':
            print(f'{mk+"/"+tk:<22}{"PRoViT-"+variant:<16}{full["n_S"]:>6}'
                  f'{"  FAILED":>36}')
            md_rows.append(f'| {mk}/{tk} | PRoViT-{variant} | {full["n_S"]} | '
                           f'FAILED | — | — | — |')
            continue
        print(f'{mk+"/"+tk:<22}{"PRoViT-"+variant:<16}{full["n_S"]:>6}'
              f'{full["repair_acc"]:>8.1f}%{full["remain_acc"]:>8.1f}%'
              f'{full["clean_acc"]:>8.1f}%{full["gen_mean"]:>8.1f}%')
        md_rows.append(f'| {mk}/{tk} | PRoViT-{variant} | {full["n_S"]} | '
                       f'{full["repair_acc"]:.1f}% | {full["remain_acc"]:.1f}% | '
                       f'{full["clean_acc"]:.1f}% | {full["gen_mean"]:.1f}% |')
    print('-' * 108)

print('\n' + '=' * 108)
print('Threshold sweep (appendix) — best General per variant subject to repair = 100%')
print('=' * 108)
print(f'{"setting":<22}{"variant":<10}{"T":>7}{"|S|":>6}{"Repair":>9}'
      f'{"Remain":>9}{"General":>9}{"AtkGen":>9}')
for mk, tk in RUN_MATRIX:
    p = f'{RESULT_DIR}/provit_{mk}_{tk}.json'
    if not os.path.exists(p): continue
    R = json.load(open(p))
    for variant in VARIANTS:
        rs = [r for r in R['variants'].get(variant, [])
              if r.get('status') == 'ok' and r.get('T') is not None]
        ok = [r for r in rs if r['repair_acc'] >= 100.0 - 1e-9] or rs
        if not ok:
            print(f'{mk+"/"+tk:<22}{variant:<10}  (all failed)'); continue
        b = max(ok, key=lambda r: r['clean_acc'])
        print(f'{mk+"/"+tk:<22}{variant:<10}{b["T"]:>7.1f}{b["n_S"]:>6}'
              f'{b["repair_acc"]:>8.1f}%{b["remain_acc"]:>8.1f}%'
              f'{b["clean_acc"]:>8.1f}%{b["gen_mean"]:>8.1f}%')

with open(f'{RESULT_DIR}/PROVIT_TABLE.md', 'w') as f:
    f.write('\n'.join(md_rows) + '\n')
print(f'\n[saved] {RESULT_DIR}/PROVIT_TABLE.md')
